# picoNewton_v4 — Google Colab runtime

Step 3 scaffold: mount Drive, create a unique run directory, clone/install the package, validate source assets, and write reproducibility manifests.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPOSITORY = 'https://github.com/khalid-saqr/picoNewton.git'
BRANCH = 'agent/piconewton-v4-step3'
PROFILE = 'quick'
!rm -rf /content/picoNewton
!git clone --branch "$BRANCH" --single-branch "$REPOSITORY" /content/picoNewton
%cd /content/picoNewton
!python -m pip install -e './picoNewton_v4[dev]'

In [ ]:
from pathlib import Path
from piconewton_v4.runtime import create_runtime
from piconewton_v4.provenance import environment_record, write_json
drive_runs = Path('/content/drive/MyDrive/picoNewton_v4_runs')
runtime = create_runtime(drive_root=drive_runs, local_base=Path('/content/piconewton_v4_runtime'), metadata={'profile': PROFILE, 'branch': BRANCH})
write_json(runtime.manifests / 'environment.json', environment_record(Path('/content/picoNewton')))
runtime.as_dict()

In [ ]:
import json
from piconewton_v4.sources import validate_cellml_sources
report = validate_cellml_sources(Path('/content/picoNewton/picoNewton_v4'))
print(json.dumps(report, indent=2))
assert report['ok'], report

In [ ]:
!pytest /content/picoNewton/picoNewton_v4/tests

In [ ]:
from datetime import datetime, timezone
write_json(runtime.manifests / 'step3_completion.json', {'step': 3, 'status': 'passed', 'completed_utc': datetime.now(timezone.utc).isoformat(), 'profile': PROFILE})
print(runtime.persistent_root)